In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import tensorflow as tf

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import contractions

In [ ]:
train = pd.read_csv(r"/content/train.csv")
test = pd.read_csv(r"/content/test.csv")

In [ ]:
print("Label distribution:")
print(train["review"].value_counts())

In [ ]:
STOPWORDS = set(stopwords.words('english'))
NEGATIONS = {"no","not","nor","never","n't"}
STOPWORDS = STOPWORDS.difference(NEGATIONS)

lemmatizer = WordNetLemmatizer()


In [ ]:
def clean_text(text, remove_stopwords=True, lemmatize=False):
    if not isinstance(text, str):
        text = str(text)

    # 1. lowercase
    text = text.lower()

    # 2. expand contractions (don't -> do not)
    text = contractions.fix(text)

    # 3. remove URLs, emails, html tags, phone numbers
    text = re.sub(r'http\S+|www\.\S+', ' ', text)            # urls
    text = re.sub(r'\S+@\S+\.\S+', ' ', text)               # emails
    text = re.sub(r'<.*?>', ' ', text)                      # html tags
    text = re.sub(r'\+?\d[\d\-\s]{5,}\d', ' ', text)        # phone-like numbers

    # 4. remove punctuation and numbers (keep letters and spaces)
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 5. reduce repeated characters: goooood -> good (limit repeats to 2)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 6. remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # 7. optional stopwords removal (but preserve negations)
    tokens = text.split()
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS]

    # 8. optional lemmatize
    if lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return ' '.join(tokens)


In [ ]:
# you can change lemma bool value if you want
train['clean'] = train['text'].astype(str).apply(lambda x: clean_text(x, remove_stopwords=True, lemmatize=True))
test['clean'] = test['text'].astype(str).apply(lambda x: clean_text(x, remove_stopwords=True, lemmatize=True))

In [ ]:
MAX_WORDS = 30000
MAX_LEN = 200

tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(train['clean'])

train_seq = tokenizer.texts_to_sequences(train['clean'])
train_padded = pad_sequences(train_seq, maxlen=MAX_LEN, padding='post')

test_seq = tokenizer.texts_to_sequences(test['clean'])
test_padded = pad_sequences(test_seq, maxlen=MAX_LEN, padding='post')

In [ ]:
le = LabelEncoder()
train['label_encoded'] = le.fit_transform(train['review'])
y = tf.keras.utils.to_categorical(train['label_encoded'], num_classes=5)

In [ ]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train['label_encoded']),
    y=train['label_encoded']
)
class_weights_dict = dict(enumerate(class_weights))
class_weights_dict